# Concurrency: Threads, Locks, Processes, asyncio, Producer–Consumer

## What you'll practice

Rule of thumb: **I/O-bound** (HTTP, S3, DB) → threads or asyncio; **CPU-bound** (parsing, hashing, compression) → processes (the GIL lets only one thread run Python bytecode at a time).
For data at scale you'll lean on Spark instead — but every DE writes a thread pool for API calls or file uploads eventually. Process-pool workers live in `labkit/workers.py` because
functions defined in a notebook can't be pickled into worker processes on Windows/macOS.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/advanced/test_10_03_concurrency_basics.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/10_advanced_python/03_concurrency_basics.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
import sys
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
sys.path.insert(0, str(LAB))
from labkit import workers      # sha256_path, count_primes, line_count

## Exercise 1 — 🟢 Easy · Ordered parallel map with threads

**Implement:** `thread_map(func, items, max_workers: int = 8) -> list`

Apply `func` to every item using a `ThreadPoolExecutor`, returning results **in input order**. The first exception raised by any call should propagate to the caller.

**Input:** `func, items, max_workers: int = 8`

**Expected output:** `list`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_thread_map_order_and_speed, test_thread_map_exception` in `tests/advanced/test_10_03_concurrency_basics.py`.

**Check:** `pytest tests/advanced/test_10_03_concurrency_basics.py -k "test_thread_map_order_and_speed or test_thread_map_exception"`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Race conditions and locks

**Implement:** `class SafeCounter`, `count_events_concurrently(batches: list[list[str]], workers: int = 8) -> dict[str, int]`

`SafeCounter` has `add(key, n=1)` and `snapshot()` (a dict copy) protected by a `threading.Lock`. `count_events_concurrently` counts event types across batches using a thread pool
writing into one shared `SafeCounter`. (Try an unlocked version with `counts[key] = counts.get(key, 0) + 1` and a `time.sleep(0)` between read and write to watch updates get lost.)

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_safe_counter_threads, test_count_events_concurrently` in `tests/advanced/test_10_03_concurrency_basics.py`.

**Check:** `pytest tests/advanced/test_10_03_concurrency_basics.py -k "test_safe_counter_threads or test_count_events_concurrently"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · CPU-bound work with processes

**Implement:** `parallel_checksums(paths: list, workers: int = 4) -> dict[str, str]`

Hash files in parallel with `ProcessPoolExecutor` using the importable worker `labkit.workers.sha256_path` (returns `(path, digest)`). Return `{str(path): digest}`. Use `chunksize`
in `pool.map` for many small files. Then compare wall time for `workers.count_primes(60_000)` × 8 tasks with threads vs processes.

**Input:** `paths: list, workers: int = 4`

**Expected output:** `dict[str, str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
parallel_checksums([])
# -> {}
```

**Check:** `pytest tests/advanced/test_10_03_concurrency_basics.py -k test_parallel_checksums`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · asyncio with bounded concurrency

**Implement:** `async fetch_all(fetch, ids: list, concurrency: int = 10) -> list`

`fetch` is an **async** function `fetch(id) -> result`. Run all fetches concurrently but never more than `concurrency` at once (`asyncio.Semaphore`), collect results **in input order**
with `asyncio.gather`. If a fetch raises, put the exception object in that slot instead of failing everything (`return_exceptions=True`). In Jupyter call it with `await fetch_all(...)`;
in scripts with `asyncio.run(fetch_all(...))`.

**Input:** `fetch, ids: list, concurrency: int = 10`

**Expected output:** `list`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_fetch_all_bounded` in `tests/advanced/test_10_03_concurrency_basics.py`.

**Check:** `pytest tests/advanced/test_10_03_concurrency_basics.py -k test_fetch_all_bounded`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Producer–consumer pipeline with bounded queues · 🧩 Mini-project

**Implement:** `run_threaded_pipeline(items, stages: list, workers_per_stage: int = 4, queue_size: int = 100) -> tuple[list, list]`

Build a multi-stage pipeline: each stage is a function `item -> item` (e.g. download → parse → upload) run by `workers_per_stage` threads, connected by `queue.Queue(maxsize=queue_size)`
(bounded = **backpressure**: a fast stage blocks instead of buffering everything in memory). A failure in any stage records `(original_item, stage_index, "Type: msg")` and drops the item.
Shut down cleanly with **sentinel ("poison pill")** objects: when all items are fed, send one sentinel per worker of stage 0; when the last worker of a stage exits, send sentinels downstream.
Return `(results, errors)` — results in any order. No thread may be left running.

**Input:** `items, stages: list, workers_per_stage: int = 4, queue_size: int = 100`

**Expected output:** `tuple[list, list]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_threaded_pipeline, test_threaded_pipeline_empty_and_single_stage` in `tests/advanced/test_10_03_concurrency_basics.py`.

**Check:** `pytest tests/advanced/test_10_03_concurrency_basics.py -k "test_threaded_pipeline or test_threaded_pipeline_empty_and_single_stage"`

**YOUR CODE ↓**